# Auditoría inicial del dataset

Este notebook documenta la revisión inicial de los archivos utilizados en el análisis de abandono de clientes.

En esta primera fase se comprobará la ubicación de los datos, se realizará un inventario de archivos y se verificará su estructura antes de efectuar transformaciones o entrenar modelos.

In [ ]:
from pathlib import Path

RUTA_CUADERNO = Path.cwd()
RUTA_PROYECTO = RUTA_CUADERNO.parent
RUTA_DATOS = RUTA_PROYECTO / "datos" / "Dataset"

print("Carpeta del cuaderno:", RUTA_CUADERNO)
print("Carpeta del proyecto:", RUTA_PROYECTO)
print("Carpeta del dataset:", RUTA_DATOS)
print("¿Existe la carpeta del dataset?", RUTA_DATOS.exists())

## Inventario inicial de archivos

Antes de cargar los datos se realiza un inventario de los archivos y subcarpetas disponibles. Esta comprobación permite identificar las fuentes de información, conocer su tamaño y decidir qué archivos deben analizarse en cada fase.

En este paso todavía no se abre el contenido de los archivos, por lo que el consumo de memoria es mínimo.

In [ ]:
elementos = sorted(RUTA_DATOS.iterdir())

print(f"Elementos encontrados: {len(elementos)}\n")

for elemento in elementos:
    if elemento.is_file():
        tamaño_mb = elemento.stat().st_size / (1024 ** 2)
        print(
            f"ARCHIVO | {elemento.name:<25} | "
            f"{tamaño_mb:,.2f} MB"
        )
    elif elemento.is_dir():
        print(f"CARPETA | {elemento.name}")

## Inventario detallado de las subcarpetas

A continuación se revisa el contenido de las subcarpetas `raw_dump`, `test`, `train` y `usage_data`.

El objetivo es identificar los archivos disponibles, conocer su tamaño y comprender cómo se distribuyen los datos antes de realizar cualquier carga completa.

In [ ]:
subcarpetas = ["raw_dump", "test", "train", "usage_data"]

for nombre_subcarpeta in subcarpetas:
    ruta_subcarpeta = RUTA_DATOS / nombre_subcarpeta
    elementos_subcarpeta = sorted(ruta_subcarpeta.iterdir())

    print("=" * 70)
    print(f"SUBCARPETA: {nombre_subcarpeta}")
    print(f"Ruta: {ruta_subcarpeta}")
    print(f"Elementos encontrados: {len(elementos_subcarpeta)}")
    print("-" * 70)

    for elemento in elementos_subcarpeta:
        if elemento.is_file():
            tamaño_mb = elemento.stat().st_size / (1024 ** 2)
            print(
                f"ARCHIVO | {elemento.name:<40} | "
                f"{tamaño_mb:,.2f} MB"
            )
        elif elemento.is_dir():
            print(f"CARPETA | {elemento.name}")

    print()

## Inspección preliminar de los archivos principales

Se examinan las primeras filas de `main.csv`, `train_cxid.csv` y `test_cxid.csv` para identificar sus columnas, tipos de datos y estructura general antes de realizar una carga completa.

La lectura se limita a cinco filas de cada archivo, por lo que el consumo de memoria es reducido.

In [ ]:
import pandas as pd

archivos_principales = {
    "main.csv": RUTA_DATOS / "main.csv",
    "train_cxid.csv": RUTA_DATOS / "train" / "train_cxid.csv",
    "test_cxid.csv": RUTA_DATOS / "test" / "test_cxid.csv",
}

muestras = {}

for nombre, ruta in archivos_principales.items():
    muestra = pd.read_csv(ruta, nrows=5)
    muestras[nombre] = muestra

    print("=" * 80)
    print(f"ARCHIVO: {nombre}")
    print(f"Ruta: {ruta}")
    print(f"Filas leídas: {len(muestra)}")
    print(f"Número de columnas: {muestra.shape[1]}")

    if muestra.shape[1] <= 15:
        print("Columnas:")
        print(muestra.columns.tolist())
    else:
        print("Primeras 10 columnas:")
        print(muestra.columns[:10].tolist())
        print("Últimas 5 columnas:")
        print(muestra.columns[-5:].tolist())

    print("\nTipos de datos detectados:")
    print(muestra.dtypes)
    print()

## Recuento de filas mediante lectura por bloques

Se determina el número exacto de registros de los archivos principales sin cargarlos completamente en memoria.

Para ello, los CSV se procesan por bloques de filas. Esta estrategia reduce el consumo de memoria y permite trabajar de forma segura con archivos de gran tamaño.

In [ ]:
def contar_filas_csv(ruta, tamaño_bloque=100_000):
    total_filas = 0

    for bloque in pd.read_csv(ruta, chunksize=tamaño_bloque):
        total_filas += len(bloque)

    return total_filas


recuento_filas = {}

for nombre, ruta in archivos_principales.items():
    total = contar_filas_csv(ruta)
    recuento_filas[nombre] = total
    print(f"{nombre:<20} | {total:>10,} filas")

## Comprobación de los conjuntos de entrenamiento y prueba

Se revisan los identificadores de clientes y la variable objetivo `churn` en los conjuntos de entrenamiento y prueba.

El objetivo es comprobar que cada cliente aparece una sola vez, que no existen valores nulos, que los conjuntos son independientes y que la variable objetivo contiene valores válidos.

In [ ]:
ruta_train = archivos_principales["train_cxid.csv"]
ruta_test = archivos_principales["test_cxid.csv"]

train_ids = pd.read_csv(ruta_train)
test_ids = pd.read_csv(ruta_test)

resumen_particiones = pd.DataFrame({
    "conjunto": ["train", "test"],
    "filas": [
        len(train_ids),
        len(test_ids)
    ],
    "clientes_unicos": [
        train_ids["customer_id"].nunique(),
        test_ids["customer_id"].nunique()
    ],
    "duplicados_customer_id": [
        train_ids["customer_id"].duplicated().sum(),
        test_ids["customer_id"].duplicated().sum()
    ],
    "nulos_customer_id": [
        train_ids["customer_id"].isna().sum(),
        test_ids["customer_id"].isna().sum()
    ],
    "nulos_churn": [
        train_ids["churn"].isna().sum(),
        test_ids["churn"].isna().sum()
    ]
})

display(resumen_particiones)

clientes_train = set(train_ids["customer_id"])
clientes_test = set(test_ids["customer_id"])
solapamiento = clientes_train.intersection(clientes_test)

print(
    "Clientes presentes simultáneamente en train y test:",
    len(solapamiento)
)

print("\nDistribución de churn en train:")
print(
    train_ids["churn"]
    .value_counts(dropna=False)
    .sort_index()
)

print("\nDistribución de churn en test:")
print(
    test_ids["churn"]
    .value_counts(dropna=False)
    .sort_index()
)

print("\nValores únicos de churn en train:")
print(sorted(train_ids["churn"].dropna().unique().tolist()))

print("\nValores únicos de churn en test:")
print(sorted(test_ids["churn"].dropna().unique().tolist()))

## Auditoría estructural y de calidad de `main.csv`

Se analiza el archivo principal mediante lectura por bloques para comprobar su estructura y detectar posibles problemas de calidad.

La revisión incluye clientes únicos, tipos de uso, valores ausentes, valores negativos, valores infinitos y combinaciones duplicadas de cliente y perfil. La lectura por bloques limita el consumo de memoria.

In [ ]:
import numpy as np

ruta_main = archivos_principales["main.csv"]
tamaño_bloque = 50_000

total_filas = 0
clientes_unicos = set()
tipos_uso = set()

nulos_customer_id = 0
nulos_usage_type = 0
nulos_dias = 0
valores_negativos = 0
valores_infinitos = 0

pares_vistos = set()
pares_duplicados = 0

columnas_dias = [f"Day_{dia}" for dia in range(1, 91)]

for numero_bloque, bloque in enumerate(
    pd.read_csv(ruta_main, chunksize=tamaño_bloque),
    start=1
):
    total_filas += len(bloque)

    clientes_unicos.update(
        bloque["customer_id"].dropna().unique()
    )
    tipos_uso.update(
        bloque["usage_type"].dropna().unique()
    )

    nulos_customer_id += bloque["customer_id"].isna().sum()
    nulos_usage_type += bloque["usage_type"].isna().sum()
    nulos_dias += bloque[columnas_dias].isna().sum().sum()

    valores = bloque[columnas_dias].to_numpy()

    valores_negativos += np.sum(valores < 0)
    valores_infinitos += np.isinf(valores).sum()

    hashes_pares = pd.util.hash_pandas_object(
        bloque[["customer_id", "usage_type"]],
        index=False
    ).to_numpy()

    for hash_par in hashes_pares:
        if hash_par in pares_vistos:
            pares_duplicados += 1
        else:
            pares_vistos.add(hash_par)

    print(
        f"Bloque {numero_bloque:02d} procesado | "
        f"filas acumuladas: {total_filas:,}"
    )

print("\n" + "=" * 65)
print("RESUMEN DE LA AUDITORÍA DE main.csv")
print("=" * 65)
print(f"Filas totales: {total_filas:,}")
print(f"Clientes únicos: {len(clientes_unicos):,}")
print(f"Tipos de uso distintos: {len(tipos_uso)}")
print(f"Nulos en customer_id: {nulos_customer_id:,}")
print(f"Nulos en usage_type: {nulos_usage_type:,}")
print(f"Nulos en columnas diarias: {nulos_dias:,}")
print(f"Valores negativos: {valores_negativos:,}")
print(f"Valores infinitos: {valores_infinitos:,}")
print(
    "Combinaciones customer_id + usage_type duplicadas:",
    f"{pares_duplicados:,}"
)

print("\nTipos de uso encontrados:")
for tipo in sorted(tipos_uso):
    print(f"  - {tipo}")

## Comprobación de la estructura de perfiles por cliente

Se comprueba que cada cliente dispone de los doce perfiles de uso esperados y que cada perfil aparece para el mismo número de clientes.

Esta validación es necesaria porque la posterior transformación del dataset presupone una estructura homogénea: doce filas de actividad por cliente y una única fila para cada combinación de cliente y tipo de uso.

In [ ]:
from collections import Counter

conteo_filas_por_cliente = Counter()
conteo_por_tipo_uso = Counter()

for bloque in pd.read_csv(
    ruta_main,
    usecols=["customer_id", "usage_type"],
    chunksize=100_000
):
    conteo_filas_por_cliente.update(
        bloque["customer_id"].value_counts().to_dict()
    )

    conteo_por_tipo_uso.update(
        bloque["usage_type"].value_counts().to_dict()
    )

distribucion_numero_perfiles = pd.Series(
    conteo_filas_por_cliente
).value_counts().sort_index()

tabla_tipos_uso = (
    pd.Series(conteo_por_tipo_uso, name="numero_registros")
    .rename_axis("usage_type")
    .reset_index()
    .sort_values("usage_type")
    .reset_index(drop=True)
)

print("Distribución del número de perfiles por cliente:")
display(
    distribucion_numero_perfiles
    .rename_axis("perfiles_por_cliente")
    .reset_index(name="numero_clientes")
)

print("\nNúmero de registros por tipo de uso:")
display(tabla_tipos_uso)

clientes_con_12_perfiles = sum(
    cantidad == 12
    for cantidad in conteo_filas_por_cliente.values()
)

clientes_con_otro_numero = sum(
    cantidad != 12
    for cantidad in conteo_filas_por_cliente.values()
)

print(f"Clientes con exactamente 12 perfiles: {clientes_con_12_perfiles:,}")
print(f"Clientes con un número distinto de 12 perfiles: {clientes_con_otro_numero:,}")

todos_tipos_completos = all(
    cantidad == len(conteo_filas_por_cliente)
    for cantidad in conteo_por_tipo_uso.values()
)

print(
    "¿Cada tipo de uso aparece para todos los clientes?",
    todos_tipos_completos
)

## Auditoría estructural de los archivos de uso

Se revisan individualmente los archivos CSV almacenados en la carpeta `usage_data`.

El objetivo es comprobar que presentan una estructura homogénea, contienen los identificadores esperados y disponen de las 90 columnas diarias necesarias para su posterior contraste con `main.csv`.

In [ ]:
ruta_usage_data = RUTA_DATOS / "usage_data"

archivos_usage_csv = sorted(
    ruta_usage_data.glob("*.csv")
)

print(f"Archivos CSV encontrados: {len(archivos_usage_csv)}")

resumen_usage = []

for ruta_archivo in archivos_usage_csv:
    cabecera = pd.read_csv(ruta_archivo, nrows=5)

    columnas = cabecera.columns.tolist()
    tiene_customer_id = "customer_id" in columnas

    columnas_dias_archivo = [
        columna
        for columna in columnas
        if columna.startswith("Day_")
    ]

    total_filas_archivo = 0
    duplicados_customer_id = 0
    nulos_customer_id = 0
    nulos_dias = 0

    ids_vistos = set()

    for bloque in pd.read_csv(
        ruta_archivo,
        chunksize=50_000
    ):
        total_filas_archivo += len(bloque)

        nulos_customer_id += (
            bloque["customer_id"].isna().sum()
        )

        nulos_dias += (
            bloque[columnas_dias_archivo]
            .isna()
            .sum()
            .sum()
        )

        for customer_id in bloque["customer_id"]:
            if customer_id in ids_vistos:
                duplicados_customer_id += 1
            else:
                ids_vistos.add(customer_id)

    resumen_usage.append({
        "archivo": ruta_archivo.name,
        "filas": total_filas_archivo,
        "columnas": len(columnas),
        "columnas_dias": len(columnas_dias_archivo),
        "tiene_customer_id": tiene_customer_id,
        "clientes_unicos": len(ids_vistos),
        "duplicados_customer_id": duplicados_customer_id,
        "nulos_customer_id": nulos_customer_id,
        "nulos_dias": nulos_dias
    })

resumen_usage_df = pd.DataFrame(resumen_usage)

display(resumen_usage_df)

## Verificación de las columnas de los archivos de uso

Tras comprobar la homogeneidad estructural de los archivos, se identifican sus columnas no diarias.

El objetivo es determinar qué información adicional acompaña a `customer_id` y confirmar que todos los archivos presentan exactamente la misma estructura.

In [ ]:
resumen_columnas_usage = []

for ruta_archivo in archivos_usage_csv:
    cabecera = pd.read_csv(ruta_archivo, nrows=0)
    columnas = cabecera.columns.tolist()

    columnas_no_diarias = [
        columna
        for columna in columnas
        if not columna.startswith("Day_")
    ]

    resumen_columnas_usage.append({
        "archivo": ruta_archivo.name,
        "numero_columnas": len(columnas),
        "columnas_no_diarias": columnas_no_diarias
    })

resumen_columnas_usage_df = pd.DataFrame(
    resumen_columnas_usage
)

display(resumen_columnas_usage_df)

estructuras_columnas = {
    tuple(pd.read_csv(ruta, nrows=0).columns)
    for ruta in archivos_usage_csv
}

print(
    "Número de estructuras de columnas diferentes:",
    len(estructuras_columnas)
)

print(
    "¿Todos los archivos tienen exactamente las mismas columnas?",
    len(estructuras_columnas) == 1
)

## Correspondencia entre archivos y tipos de uso

Se comprueba que cada archivo de la carpeta `usage_data` contiene un único valor de `usage_type` y que dicho valor coincide con el perfil indicado en el nombre del archivo.

También se contrasta que el conjunto de tipos de uso de los archivos individuales coincide con los doce perfiles identificados en `main.csv`.

In [ ]:
resumen_correspondencia = []

tipos_en_archivos = set()

for ruta_archivo in archivos_usage_csv:
    valores_tipo = set()

    for bloque in pd.read_csv(
        ruta_archivo,
        usecols=["usage_type"],
        chunksize=50_000
    ):
        valores_tipo.update(
            bloque["usage_type"].dropna().unique()
        )

    tipos_en_archivos.update(valores_tipo)

    nombre_sin_extension = ruta_archivo.stem

    resumen_correspondencia.append({
        "archivo": ruta_archivo.name,
        "numero_tipos_en_archivo": len(valores_tipo),
        "tipos_encontrados": sorted(valores_tipo),
        "coincide_con_nombre_archivo": (
            valores_tipo == {nombre_sin_extension}
        )
    })

resumen_correspondencia_df = pd.DataFrame(
    resumen_correspondencia
)

display(resumen_correspondencia_df)

print(
    "¿Cada archivo contiene un único tipo de uso?",
    resumen_correspondencia_df[
        "numero_tipos_en_archivo"
    ].eq(1).all()
)

print(
    "¿El tipo de uso coincide con el nombre de cada archivo?",
    resumen_correspondencia_df[
        "coincide_con_nombre_archivo"
    ].all()
)

print(
    "¿Los tipos de los archivos coinciden con los de main.csv?",
    tipos_en_archivos == tipos_uso
)

print("\nTipos presentes en los archivos individuales:")
for tipo in sorted(tipos_en_archivos):
    print(f"  - {tipo}")

## Contraste entre `main.csv` y los archivos individuales de uso

Se comprueba que cada archivo de `usage_data` contiene exactamente los mismos registros y valores diarios que el subconjunto correspondiente de `main.csv`.

La comparación se realiza por tipo de uso, cliente y columnas diarias, sin cargar simultáneamente todo el archivo principal en memoria.

In [ ]:
from collections import defaultdict

columnas_comparacion = [
    "customer_id",
    "usage_type",
    *columnas_dias
]

hashes_main_por_tipo = defaultdict(set)

for bloque in pd.read_csv(
    ruta_main,
    usecols=columnas_comparacion,
    chunksize=50_000
):
    for tipo, grupo in bloque.groupby("usage_type"):
        hashes = pd.util.hash_pandas_object(
            grupo[columnas_comparacion],
            index=False
        ).to_numpy()

        hashes_main_por_tipo[tipo].update(hashes)

resultados_comparacion = []

for ruta_archivo in archivos_usage_csv:
    tipo_archivo = ruta_archivo.stem
    hashes_archivo = set()

    for bloque in pd.read_csv(
        ruta_archivo,
        usecols=columnas_comparacion,
        chunksize=50_000
    ):
        hashes = pd.util.hash_pandas_object(
            bloque[columnas_comparacion],
            index=False
        ).to_numpy()

        hashes_archivo.update(hashes)

    hashes_main = hashes_main_por_tipo[tipo_archivo]

    resultados_comparacion.append({
        "usage_type": tipo_archivo,
        "registros_main": len(hashes_main),
        "registros_archivo": len(hashes_archivo),
        "faltan_en_archivo": len(hashes_main - hashes_archivo),
        "sobran_en_archivo": len(hashes_archivo - hashes_main),
        "contenido_identico": hashes_main == hashes_archivo
    })

comparacion_usage_df = pd.DataFrame(
    resultados_comparacion
)

display(comparacion_usage_df)

print(
    "¿Todos los archivos individuales coinciden con main.csv?",
    comparacion_usage_df["contenido_identico"].all()
)